# Flappy Bird PPO — Google Colab Version

Adapted from the local notebook. Key differences for Colab:
- No real display, so we use a **virtual display (Xvfb)** for rendering.
- `render_mode="human"` is replaced with `render_mode="rgb_array"` + video recording, since Colab can't open a live window.
- Model is saved to Google Drive so it survives runtime disconnects.

## 1. Install dependencies

In [ ]:
!pip install -q flappy-bird-gymnasium stable-baselines3
!pip install -q gymnasium[classic-control]
!apt-get install -y xvfb > /dev/null 2>&1
!pip install -q pyvirtualdisplay

## 2. Start a virtual display

Creates a fake screen buffer so pygame/gymnasium can render frames without a real monitor.

In [ ]:
from pyvirtualdisplay import Display

display = Display(visible=0, size=(400, 300))
display.start()

## 3. Understand the environment

Same as local — just confirming action/observation spaces.

In [ ]:
import gymnasium as gym
import flappy_bird_gymnasium

env = gym.make("FlappyBird-v0", render_mode=None)
print("Action space:", env.action_space)
print("Observation space:", env.observation_space)
env.close()

## 4. Train the PPO model

No rendering needed during training, so this is identical to the local version.
On Colab's free-tier CPU, ~150,000 timesteps should take roughly 8–10 minutes.

In [ ]:
from stable_baselines3 import PPO

env = gym.make("FlappyBird-v0", render_mode=None)

model = PPO('MlpPolicy', env, verbose=1)
model.learn(total_timesteps=150000)

model.save("ppo_flappybird")
env.close()

## 5. Evaluate with video recording

Instead of `render_mode="human"`, we use `render_mode="rgb_array"` wrapped in `RecordVideo`
so we get an .mp4 we can watch directly in the notebook.

In [ ]:
from gymnasium.wrappers import RecordVideo
from stable_baselines3.common.evaluation import evaluate_policy

eval_env = gym.make("FlappyBird-v0", render_mode="rgb_array")
eval_env = RecordVideo(eval_env, video_folder="videos", episode_trigger=lambda x: True)

mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=5)
eval_env.close()

print(f"Mean reward: {mean_reward} +/- {std_reward}")

## 6. Play the recorded video inline

In [ ]:
import os

video_dir = "videos"
fixed_dir = "videos_fixed"
os.makedirs(fixed_dir, exist_ok=True)

for f in sorted(os.listdir(video_dir)):
    if f.endswith(".mp4"):
        src = os.path.join(video_dir, f)
        size = os.path.getsize(src)
        if size < 10_000:  # skip genuinely broken/empty files
            print(f"Skipping {f} (only {size} bytes — likely empty episode)")
            continue
        dst = os.path.join(fixed_dir, f)
        !ffmpeg -y -loglevel error -i "{src}" -c copy "{dst}"
        print(f"Fixed: {f}")

In [ ]:
import os

video_dir = "videos"
fixed_dir = "videos_fixed"
os.makedirs(fixed_dir, exist_ok=True)

for f in sorted(os.listdir(video_dir)):
    if f.endswith(".mp4"):
        src = os.path.join(video_dir, f)
        size = os.path.getsize(src)
        dst = os.path.join(fixed_dir, f)
        !ffmpeg -y -loglevel error -i "{src}" -c copy "{dst}"
        print(f"Fixed: {f}")

In [ ]:
from IPython.display import Video, display

fixed_files = sorted([f for f in os.listdir(fixed_dir) if f.endswith(".mp4")])
print("Fixed videos:", fixed_files)

display(Video(os.path.join(fixed_dir, fixed_files[0]), embed=True, width=400))

## 7. Persist the model to Google Drive

Colab wipes its filesystem when the runtime disconnects — save the model to Drive so it's not lost.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
!cp ppo_flappybird.zip /content/drive/MyDrive/ppo_flappybird.zip
print("Saved to /content/drive/MyDrive/ppo_flappybird.zip")

## 8. (Later) Reload the model without retraining

In [ ]:
# Run this instead of the training cell if you already have a saved model in Drive

# from google.colab import drive
# drive.mount('/content/drive')
# from stable_baselines3 import PPO
# model = PPO.load('/content/drive/MyDrive/ppo_flappybird')